In [ ]:
# Import libraries
import random
import math
import torch
import torch.nn as nn
from torch.nn import functional as F
from transformers import LlamaTokenizer
from datasets import load_dataset

import warnings
warnings.filterwarnings("ignore")

# 1) Initialization

In [2]:
# Define model ID and load tokenizer
model_id = "daryl149/llama-2-7b-chat-hf"
tokenizer = LlamaTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Define model configuration parameters
config = {
    "vocab_size": tokenizer.vocab_size,
    "n_layers": 1,
    "embed_dim": 2048,
    "n_heads": 32,
    "n_kv_heads": 8,
    "multiple_of": 64,
    "ffn_dim_multiplier": None,
    "norm_eps": 1e-5,
    "max_batch_size": 16,
    "max_seq_len": 64,
    "device": (
        "cuda"
        if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available() else "cpu"
    ),
}

# Load and preprocess dataset
dataset = load_dataset("nyu-mll/glue", "ax", split="test")
dataset = dataset.select_columns(["premise", "hypothesis"])

# Tokenize a sample of the dataset
# Input: list of batch_num strings
# Output: input_ids → (batch_num, seq_len), attention_mask → (batch_num, seq_len)
test_set = tokenizer(
    random.sample(dataset["premise"], config["max_batch_size"]),
    truncation=True,
    max_length=config["max_seq_len"],
    padding="max_length",
    return_tensors="pt",
)

# Print keys and size of the test set
print(test_set.keys())
print(test_set["input_ids"].size())

KeysView({'input_ids': tensor([[    2,     2,     2,  ...,   526,   451, 29889],
        [    2,     2,     2,  ..., 28026, 21904, 29889],
        [    2,     2,     2,  ...,   367,   550, 29889],
        ...,
        [    2,     2,     2,  ..., 28042, 28606, 29889],
        [    2,     2,     2,  ...,   471, 10124, 29889],
        [    2,     2,     2,  ...,  1820, 12651, 29889]]), 'attention_mask': tensor([[0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        ...,
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 1, 1, 1]])})
torch.Size([16, 64])


# 2) Llama Architecture

<img src='https://miro.medium.com/v2/resize:fit:1200/1*ZbnVUpK5pw5iJJeeiBa-9w.png' width=500>

[Concept Explained](https://akgeni.medium.com/llama-concepts-explained-summary-a87f0bd61964) | [Llama Github](https://github.com/facebookresearch/llama)

## 2.1 Rotary Embedding

<img src='https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTF1FTZO0ZaN6MFKvdcRLYBqRnup_IAKvJRQAG2MC0oMw&s=10' width=600>

In [3]:
# Precompute theta and positional frequencies for Rotary Position Embeddings (RoPE)
def precompute_theta_pos_frequencies(head_dim, seq_len, device, theta=10000.0):

    # Compute base theta values: theta_i = 10000^(-2(i-1)/head_dim) for i in [1 .. head_dim/2]
    # Output: (head_dim // 2,)
    theta_numerator = torch.arange(0, head_dim, 2).float()
    # (head_dim // 2,)
    theta = 1.0 / (theta ** (theta_numerator / head_dim)).to(device)

    # Create position index vector
    # Output: (seq_len,)
    m = torch.arange(seq_len, device=device)

    # Compute outer product of positions and theta values
    # (seq_len,) x (head_dim // 2,) → (seq_len, head_dim // 2)
    freqs = torch.outer(m, theta).float()

    # Convert to complex polar form: exp(i * freq)
    # (seq_len, head_dim // 2) complex
    freqs_complex = torch.polar(torch.ones_like(freqs), freqs)
    return freqs_complex


# Apply rotary embeddings to query or key tensor
def apply_rotary_embeddings(x, freqs_complex, device):

    # Reshape last dimension into pairs for complex number conversion
    # (batch_num, seq_len, num_heads, head_dim) → (batch_num, seq_len, num_heads, head_dim // 2, 2)
    x = x.float().reshape(*x.shape[:-1], -1, 2)

    # View pairs as complex numbers
    # (batch_num, seq_len, num_heads, head_dim // 2, 2) → (batch_num, seq_len, num_heads, head_dim // 2) complex
    x_complex = torch.view_as_complex(x)

    # Broadcast freqs to match batch and head dimensions
    # (seq_len, head_dim // 2) → (1, seq_len, 1, head_dim // 2) complex
    freqs_complex = freqs_complex.unsqueeze(0).unsqueeze(2)

    # Apply rotation via complex multiplication
    # (batch_num, seq_len, num_heads, head_dim // 2) x (1, seq_len, 1, head_dim // 2) → (batch_num, seq_len, num_heads, head_dim // 2) complex
    x_rotated = x_complex * freqs_complex

    # Convert rotated complex values back to real pairs
    # (batch_num, seq_len, num_heads, head_dim // 2) complex → (batch_num, seq_len, num_heads, head_dim // 2, 2)
    x_out = torch.view_as_real(x_rotated)

    # Flatten the last two dimensions back to head_dim
    # (batch_num, seq_len, num_heads, head_dim // 2, 2) → (batch_num, seq_len, num_heads, head_dim)
    x_out = x_out.reshape(*x.shape[:-2], -1)

    return x_out.type_as(x).to(device)

## 2.2 RMSNorm

LLaMA normalises with RMSNorm (no mean subtraction, no bias), applied **before** each
sub-layer and once more before the LM head. The statistic is computed in float32 and cast
back, which matters once activations are in fp16/bf16.

In [7]:
# Implementation of RMSNorm (Root Mean Square Layer Normalization)
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # Learnable scale parameter
        # Shape: (embed_dim,)
        self.weight = nn.Parameter(torch.ones(dim))

    # Internal normalization: scale each element by inverse RMS of its row
    def _norm(self, x: torch.Tensor):
        # Compute per-row RMS and normalize
        # (batch_num, seq_len, embed_dim)
        # rsqrt: 1 / sqrt(x)
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    # Forward pass: normalize then re-scale by learned weight
    def forward(self, x: torch.Tensor):
        # Apply RMS normalization then multiply by learnable gain
        # (batch_num, seq_len, embed_dim)
        # weight (embed_dim,) broadcasts over batch_num and seq_len
        return self.weight * self._norm(x.float()).type_as(x)

## 2.3 KV Caching
<img src="https://miro.medium.com/v2/resize:fit:1400/1*uyuyOW1VBqmF5Gtv225XHQ.gif" width=500>

In [8]:
# Implementation of KV Cache for efficient autoregressive inference
class KVCache:
    def __init__(self, max_batch_size, max_seq_len, n_kv_heads, head_dim, device):
        # Pre-allocate key cache tensor
        # Shape: (max_batch_size, max_seq_len, num_kv_heads, head_dim)
        self.cache_k = torch.zeros(
            (max_batch_size, max_seq_len, n_kv_heads, head_dim)
        ).to(device)
        # Pre-allocate value cache tensor
        # Shape: (max_batch_size, max_seq_len, num_kv_heads, head_dim)
        self.cache_v = torch.zeros(
            (max_batch_size, max_seq_len, n_kv_heads, head_dim)
        ).to(device)

    # Write new key/value entries at position start_pos in the cache
    # xk, xv: (batch_num, seq_len, num_kv_heads, head_dim)
    def update(self, batch_size, start_pos, xk, xv):
        self.cache_k[:batch_size, start_pos : start_pos + xk.size(1)] = xk
        self.cache_v[:batch_size, start_pos : start_pos + xv.size(1)] = xv

    # Retrieve cached keys and values up to current position
    # Output keys, values: (batch_num, start_pos + seq_len, num_kv_heads, head_dim)
    def get(self, batch_size, start_pos, seq_len):
        keys = self.cache_k[:batch_size, : start_pos + seq_len]
        values = self.cache_v[:batch_size, : start_pos + seq_len]
        return keys, values

## 2.4 Grouped Query Attention
<img src="https://pbs.twimg.com/media/FzjhZk5X0AYAs_-?format=jpg&name=medium" width=500>


In [9]:
# Helper: repeat KV heads to match the number of query heads (Grouped Query Attention)
def repeat_kv(x, n_rep):
    # x: (batch_num, seq_len, num_kv_heads, head_dim)
    batch_size, seq_len, n_kv_heads, head_dim = x.shape
    if n_rep == 1:
        return x
    else:
        # Expand KV heads by n_rep via broadcasting then reshape
        # Input:         (batch_num, seq_len, num_kv_heads, head_dim)
        # After unsqueeze:(batch_num, seq_len, num_kv_heads, 1, head_dim)
        # After expand:  (batch_num, seq_len, num_kv_heads, n_rep, head_dim)
        # Output reshape:(batch_num, seq_len, num_kv_heads * n_rep, head_dim)
        return (
            x[:, :, :, None, :]
            .expand(batch_size, seq_len, n_kv_heads, n_rep, head_dim)
            .reshape(batch_size, seq_len, n_kv_heads * n_rep, head_dim)
        )


# Implementation of Grouped Query Self-Attention (GQA) with RoPE and KV Cache
class SelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.n_heads = config["n_heads"]
        self.n_kv_heads = config["n_kv_heads"]
        self.dim = config["embed_dim"]
        self.n_kv_heads = self.n_heads if self.n_kv_heads is None else self.n_kv_heads
        self.n_heads_q = self.n_heads
        # Repetition factor: how many Q heads share each KV head
        self.n_rep = self.n_heads_q // self.n_kv_heads
        self.head_dim = self.dim // self.n_heads

        # Query projection: embed_dim → num_heads * head_dim
        self.wq = nn.Linear(self.dim, self.n_heads * self.head_dim, bias=False)
        # Key projection (GQA: fewer heads): embed_dim → num_kv_heads * head_dim
        self.wk = nn.Linear(self.dim, self.n_kv_heads * self.head_dim, bias=False)
        # Value projection (GQA: fewer heads): embed_dim → num_kv_heads * head_dim
        self.wv = nn.Linear(self.dim, self.n_kv_heads * self.head_dim, bias=False)
        # Output projection: num_heads * head_dim → embed_dim
        self.wo = nn.Linear(self.n_heads * self.head_dim, self.dim, bias=False)

        # Initialize KV Cache for inference
        self.cache = KVCache(
            max_batch_size=config["max_batch_size"],
            max_seq_len=config["max_seq_len"],
            n_kv_heads=self.n_kv_heads,
            head_dim=self.head_dim,
            device=config["device"],
        )

    # Forward pass for Grouped Query Self-Attention with RoPE and KV Cache
    def forward(self, x, start_pos, freqs_complex):

        # seq_len is 1 during autoregressive inference, full seq_len during training
        batch_size, seq_len, _ = x.shape

        # Project input to query space
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, num_heads * head_dim)
        xq = self.wq(x)

        # Project input to key space (GQA: fewer KV heads than Q heads)
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, num_kv_heads * head_dim)
        xk = self.wk(x)
        # Project input to value space
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, num_kv_heads * head_dim)
        xv = self.wv(x)

        # Reshape Q into separate heads
        # (batch_num, seq_len, num_heads * head_dim) → (batch_num, seq_len, num_heads, head_dim)
        xq = xq.view(batch_size, seq_len, self.n_heads_q, self.head_dim)

        # Reshape K into separate KV heads
        # (batch_num, seq_len, num_kv_heads * head_dim) → (batch_num, seq_len, num_kv_heads, head_dim)
        xk = xk.view(batch_size, seq_len, self.n_kv_heads, self.head_dim)
        # Reshape V into separate KV heads
        # (batch_num, seq_len, num_kv_heads * head_dim) → (batch_num, seq_len, num_kv_heads, head_dim)
        xv = xv.view(batch_size, seq_len, self.n_kv_heads, self.head_dim)

        # Apply Rotary Position Embeddings (RoPE) to query vectors
        # (batch_num, seq_len, num_heads, head_dim)
        xq = apply_rotary_embeddings(xq, freqs_complex, device=x.device)

        # Apply Rotary Position Embeddings (RoPE) to key vectors
        # (batch_num, seq_len, num_kv_heads, head_dim)
        xk = apply_rotary_embeddings(xk, freqs_complex, device=x.device)

        # Write current K, V into the KV cache at start_pos
        # xk, xv: (batch_num, seq_len, num_kv_heads, head_dim) stored into cache
        self.cache.update(batch_size, start_pos, xk, xv)

        # Read all accumulated K, V from cache (covers positions 0 .. start_pos + seq_len)
        # Output keys, values: (batch_num, start_pos + seq_len, num_kv_heads, head_dim)
        keys, values = self.cache.get(batch_size, start_pos, seq_len)

        # Expand KV heads to match number of Q heads (GQA → MHA-equivalent via repetition)
        # (batch_num, start_pos + seq_len, num_kv_heads, head_dim) → (batch_num, start_pos + seq_len, num_heads, head_dim)
        keys = repeat_kv(keys, self.n_rep)
        values = repeat_kv(values, self.n_rep)

        # Transpose Q to (batch_num, num_heads, seq_len, head_dim) for batched matmul
        # (batch_num, seq_len, num_heads, head_dim) → (batch_num, num_heads, seq_len, head_dim)
        xq = xq.transpose(1, 2)

        # Transpose K to (batch_num, num_heads, start_pos + seq_len, head_dim)
        # (batch_num, start_pos + seq_len, num_heads, head_dim) → (batch_num, num_heads, start_pos + seq_len, head_dim)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention scores
        # (batch_num, num_heads, seq_len, head_dim) @ (batch_num, num_heads, head_dim, start_pos + seq_len) → (batch_num, num_heads, seq_len, start_pos + seq_len)
        scores = torch.matmul(xq, keys.transpose(2, 3)) / math.sqrt(self.head_dim)

        # Causal mask: query at absolute position start_pos + i sees keys 0 .. start_pos + i
        # (seq_len, start_pos + seq_len); all zeros when decoding a single token
        mask = torch.full((seq_len, start_pos + seq_len), float("-inf"), device=x.device)
        mask = torch.triu(mask, diagonal=start_pos + 1)
        scores = scores + mask

        # Apply softmax over key-length dimension to get attention weights
        # (batch_num, num_heads, seq_len, start_pos + seq_len)
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)

        # Weighted sum of value vectors
        # (batch_num, num_heads, seq_len, start_pos + seq_len) @ (batch_num, num_heads, start_pos + seq_len, head_dim) → (batch_num, num_heads, seq_len, head_dim)
        output = torch.matmul(scores, values)

        # Merge heads and flatten back to embed_dim
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, seq_len, embed_dim)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)

        # Final output projection
        # (batch_num, seq_len, embed_dim)
        return self.wo(output)

## 2.5 SwiGlu

<img src="https://miro.medium.com/v2/resize:fit:596/format:webp/1*W1sg5_qhRfOCWOM1hkXi_Q.png" width=500>

In [10]:
# Sigmoid activation function (gating component in SwiGLU)
def sigmoid(x):
    # any shape → same shape (element-wise)
    return 1 / (1 + torch.exp(-x))


# SwiGLU activation: x * sigmoid(x) — smooth non-monotonic gating used in LLaMA FFN
def swiglu(x):
    # (batch_num, seq_len, intermediate_dim)
    return x * sigmoid(x)

## 2.6 FeedForward Layer and Decoder

In [11]:
# Implementation of the SwiGLU Feed-Forward Network (FFN)
class FeedForward(nn.Module):
    def __init__(self, config):

        super().__init__()

        # Compute intermediate_dim using LLaMA formula: 2/3 * 4 * embed_dim
        hidden_dim = 4 * config["embed_dim"]
        hidden_dim = int(2 * hidden_dim / 3)

        if config["ffn_dim_multiplier"] is not None:
            hidden_dim = int(config["ffn_dim_multiplier"] * hidden_dim)

        # Round intermediate_dim up to the nearest multiple_of for hardware efficiency
        hidden_dim = config["multiple_of"] * (
            (hidden_dim + config["multiple_of"] - 1) // config["multiple_of"]
        )

        # Gate projection (w1): embed_dim → intermediate_dim
        self.w1 = nn.Linear(config["embed_dim"], hidden_dim, bias=False)
        # Up projection (w2): embed_dim → intermediate_dim  (second SwiGLU branch)
        self.w2 = nn.Linear(config["embed_dim"], hidden_dim, bias=False)
        # Down projection (w3): intermediate_dim → embed_dim
        self.w3 = nn.Linear(hidden_dim, config["embed_dim"], bias=False)

    # Forward pass: SwiGLU(gate) * up → down-project
    def forward(self, x: torch.Tensor):
        # Gate branch: apply SwiGLU to w1 projection
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, intermediate_dim)
        swish = swiglu(self.w1(x))
        # Up branch: linear projection without activation
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, intermediate_dim)
        x_V = self.w2(x)

        # Element-wise multiply gate and up branches (SwiGLU gating mechanism)
        # (batch_num, seq_len, intermediate_dim) * (batch_num, seq_len, intermediate_dim) → (batch_num, seq_len, intermediate_dim)
        x = swish * x_V

        # Down projection back to model dimension
        # (batch_num, seq_len, intermediate_dim) → (batch_num, seq_len, embed_dim)
        return self.w3(x)

In [12]:
# Implementation of a single LLaMA Decoder Block (pre-norm architecture)
class DecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.n_heads = config["n_heads"]
        self.dim = config["embed_dim"]
        self.head_dim = self.dim // self.n_heads

        # Sub-layers: grouped query self-attention and SwiGLU feed-forward
        self.attention = SelfAttention(config)
        self.feed_forward = FeedForward(config)

        # Pre-norm RMSNorm applied before attention
        # Weight shape: (embed_dim,)
        self.attention_norm = RMSNorm(self.dim, eps=config["norm_eps"])

        # Pre-norm RMSNorm applied before feed-forward
        # Weight shape: (embed_dim,)
        self.ffn_norm = RMSNorm(self.dim, eps=config["norm_eps"])

    # Forward pass: pre-norm → attention + residual, then pre-norm → FFN + residual
    def forward(self, x, start_pos, freqs_complex):

        # Apply RMSNorm, self-attention, then add residual
        # attention_norm: (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        # attention output: (batch_num, seq_len, embed_dim)
        # h = x + attn_out: (batch_num, seq_len, embed_dim)
        h = x + self.attention.forward(self.attention_norm(x), start_pos, freqs_complex)

        # Apply RMSNorm, feed-forward, then add residual
        # ffn_norm: (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        # ffn output: (batch_num, seq_len, embed_dim)
        # out = h + ffn_out: (batch_num, seq_len, embed_dim)
        out = h + self.feed_forward.forward(self.ffn_norm(h))
        return out

In [13]:
# Full LLaMA Transformer: embedding → num_layers decoder blocks → RMSNorm → LM head
class Transformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab_size = config["vocab_size"]
        self.n_layers = config["n_layers"]

        # Token embedding lookup table
        # Weight shape: (vocab_size, embed_dim)
        self.tok_embeddings = nn.Embedding(self.vocab_size, config["embed_dim"])
        self.head_dim = config["embed_dim"] // config["n_heads"]

        # Stack of num_layers decoder blocks
        self.layers = nn.ModuleList()
        for layer_id in range(config["n_layers"]):
            self.layers.append(DecoderBlock(config))

        # Final RMSNorm applied before the LM head
        # Weight shape: (embed_dim,)
        self.norm = RMSNorm(config["embed_dim"], eps=config["norm_eps"])

        # Language model head: project hidden states to vocabulary logits
        # Weight shape: (vocab_size, embed_dim)
        self.output = nn.Linear(config["embed_dim"], self.vocab_size, bias=False)

        # Precompute RoPE frequency tensor for all positions
        # Output shape: (max_seq_len * 2, head_dim // 2) complex
        self.freqs_complex = precompute_theta_pos_frequencies(
            self.head_dim, config["max_seq_len"] * 2, device=(config["device"])
        )

    # Forward pass: token ids → vocabulary logits
    def forward(self, tokens, start_pos):
        # Unpack input shape
        # tokens: (batch_num, seq_len)
        batch_size, seq_len = tokens.shape

        # Embed token ids into dense vectors
        # (batch_num, seq_len) → (batch_num, seq_len, embed_dim)
        h = self.tok_embeddings(tokens)

        # Slice precomputed RoPE frequencies for current position range
        # Output: (seq_len, head_dim // 2) complex
        freqs_complex = self.freqs_complex[start_pos : start_pos + seq_len]

        # Pass through each decoder block sequentially
        # Per block: (batch_num, seq_len, embed_dim)
        for layer in self.layers:
            h = layer(h, start_pos, freqs_complex)

        # Apply final RMSNorm
        # (batch_num, seq_len, embed_dim)
        h = self.norm(h)

        # Project to vocabulary logits
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, vocab_size)
        output = self.output(h).float()
        return output


# Initialize Transformer model and run a forward pass on the tokenized test set
model = Transformer(config).to(config["device"])
# (batch_num, seq_len) → (batch_num, seq_len, vocab_size)
res = model.forward(test_set["input_ids"].to(config["device"]), 0)
print(res.size())

torch.Size([16, 64, 32000])


In [ ]:
# Causality + KV-cache check at a middle position t: a full forward pass must give the
# same logits as prefilling tokens 0 .. t-1 and then decoding token t from the cache
input_ids = test_set["input_ids"][:2].to(config["device"])
t = input_ids.size(1) // 2
with torch.no_grad():
    full_logits = model(input_ids, 0)[:, t]
    model(input_ids[:, :t], 0)
    step_logits = model(input_ids[:, t : t + 1], t)[:, -1]
print(f"Max |full - cached| logit diff at t={t}: {(full_logits - step_logits).abs().max():.2e}")